In [0]:
%python
import pandas as pd

# --- Read source CSV from landing zone ---
src_path = "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/facilities/TABLE 9.18 Number of Regulated Hospitals by Type and by Region- 2020 to 2024 (1).csv"
df = pd.read_csv(src_path, sep=",", header=1)

# --- Unpivot wide format to long format ---
# Source columns: Region, Total 2020..2024, Government 2020..2024, Private 2020..2024
melted = df.melt(
    id_vars=["Region"],
    var_name="col_name",
    value_name="number_of_hospitals",
)

# Extract year (e.g. "Total 2020" -> 2020)
melted["year"] = melted["col_name"].str.extract(r"(\d{4})").astype(int)

# Extract nature of ownership (e.g. "Government 2020" -> "Government", "Total 2020" -> "total")
melted["nature_of_ownership"] = (
    melted["col_name"]
    .str.replace(r"\s*\d{4}$", "", regex=True)
    .str.lower()
)

# Filter out "total" — keep only government and private
melted = melted[melted["nature_of_ownership"] != "total"].copy()

# Rename and select final columns
melted = melted.rename(columns={"Region": "region"})
melted = melted[["region", "year", "nature_of_ownership", "number_of_hospitals"]]
melted = melted.sort_values(["region", "year", "nature_of_ownership"]).reset_index(drop=True)

# Convert all columns to nullable string for Arrow compatibility
for col in melted.columns:
    melted[col] = melted[col].astype("string")

# --- Write to Delta table ---
table_name = "public_health.bronze.facilities_regulated_hospital"
spark_df = spark.createDataFrame(melted)
(
    spark_df.write.format("delta")
    .mode("overwrite")
    .option("columnMapping.mode", "name")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print(f"Wrote {melted.shape[0]} rows to {table_name}")
display(spark.table(table_name).limit(20))